In [22]:
import numpy as np
import pandas as pd

TRAIN = 'data/labeledTrainData.tsv'
BONUS = 'data/unlabeledTrainData.tsv'
TEST = 'data/testData.tsv'

train = pd.read_csv(TRAIN, header=0, delimiter="\t", quoting=3)
bonus = pd.read_csv(BONUS, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST, header=0, delimiter="\t", quoting=3)

print(f"Train shape: {train.shape}")
print(f"Bonus shape: {bonus.shape}")
print(f"Test shape: {test.shape}")

print(train.head())

Train shape: (25000, 3)
Bonus shape: (50000, 2)
Test shape: (25000, 2)
         id  sentiment                                             review
0  "5814_8"          1  "With all this stuff going down at the moment ...
1  "2381_9"          1  "\"The Classic War of the Worlds\" by Timothy ...
2  "7759_3"          0  "The film starts with a manager (Nicholas Bell...
3  "3630_4"          0  "It must be assumed that those who praised thi...
4  "9495_8"          1  "Superbly trashy and wondrously unpretentious ...


In [23]:
drop_cols = ['id']
train.drop(columns=drop_cols, inplace=True)
bonus.drop(columns=drop_cols, inplace=True)
test_ids = test['id']
test.drop(columns=drop_cols, inplace=True)

test_ids = [x.replace('"', '') for x in test_ids]
print(test_ids[:5])

['12311_10', '8348_2', '5828_4', '7186_2', '12128_7']


In [24]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train['review'] = train['review'].apply(clean_text)
bonus['review'] = bonus['review'].apply(clean_text)
test['review'] = test['review'].apply(clean_text)

print(train['review'][0])

with all this stuff going down at the moment with mj ive started listening to his music watching the odd documentary here and there watched the wiz and watched moonwalker again maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent moonwalker is part biography part feature film which i remember going to see at the cinema when it was originally released some of it has subtle messages about mjs feeling towards the press and also the obvious message of drugs are bad mkaybr br visually impressive but of course this is all about michael jackson so unless you remotely like mj in anyway then you are going to hate this and find it boring some may call mj an egotist for consenting to the making of this movie but mj and most of his fans would say that he made it for the fans which if true is really nice of himbr br the actual feature film bit when it finally starts is only on for 20 min

In [25]:
print(train.head(1))
print(bonus.head(1))
print(test.head(1))

   sentiment                                             review
0          1  with all this stuff going down at the moment w...
                                              review
0  watching time chasers it obvious that it was m...
                                              review
0  naturally in a film whos main themes are of mo...


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(train['review'], train['sentiment'], test_size=0.2, random_state=42)

tokens = 0
sentences = []
for review in (X_train.tolist() + bonus['review'].tolist()):
    sentences.append(review.split())
    tokens += len(sentences[-1])

print(f"Total tokens: {tokens}")
print(f"Number of sentences: {len(sentences)}")
print(f"First sentence: {sentences[0]}")

Total tokens: 16326081
Number of sentences: 70000
First sentence: ['this', 'movie', 'is', 'just', 'plain', 'dumbbr', 'br', 'from', 'the', 'casting', 'of', 'ralph', 'meeker', 'as', 'mike', 'hammer', 'to', 'the', 'fatuous', 'climax', 'the', 'film', 'is', 'an', 'exercise', 'in', 'wooden', 'predictabilitybr', 'br', 'mike', 'hammer', 'is', 'one', 'of', 'detective', 'fictions', 'true', 'sociopaths', 'unlike', 'marlow', 'and', 'spade', 'who', 'put', 'pieces', 'together', 'to', 'solve', 'the', 'mystery', 'hammer', 'breaks', 'things', 'apart', 'to', 'get', 'to', 'the', 'truth', 'this', 'film', 'turns', 'hammer', 'into', 'a', 'boob', 'by', 'surrounding', 'him', 'with', 'bad', 'guys', 'who', 'are', 'well', 'too', 'dumb', 'to', 'get', 'away', 'with', 'anything', 'one', 'is', 'so', 'poorly', 'drawn', 'that', 'he', 'succumbs', 'to', 'a', 'popcorn', 'attackbr', 'br', 'other', 'parts', 'of', 'the', 'movie', 'are', 'right', 'out', 'of', 'the', 'three', 'stooges', 'play', 'book', 'veldas', 'dance', 'at'

In [27]:
from gensim.models import FastText
import os

def sentence_vector(text, fasttext):
    words = [w for w in text.split() if w in fasttext.wv]
    if not words:
        return np.zeros(fasttext.vector_size)
    return np.mean(fasttext.wv[words], axis=0)

# fasttext = FastText(
#     sentences=sentences,
#     vector_size=300,
#     window=10,
#     min_count=5,
#     workers=os.cpu_count(),
#     sg=1,
#     epochs=10,
#     seed=42
# )
# fasttext.save('models/fasttext_v1.model')

In [28]:
# fasttext = FastText.load('models/fasttext_v1.model')

# print(fasttext.wv[['hi', 'hello']])

# X_train = np.array([sentence_vector(text) for text in X_train])
# X_val = np.array([sentence_vector(text) for text in X_val])

# print(X_train[0])

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV

# params = {
#     'n_estimators': [1600, 2000],
#     # 'max_depth': [5, 10, 20, None],
# }

# model_rf = RandomForestClassifier(
#     n_estimators=400, 
#     criterion='entropy',
#     n_jobs=-1,
#     random_state=42
# )

# grid_search = GridSearchCV(
#     estimator=model_rf,
#     param_grid=params,
#     scoring='roc_auc',
#     cv=3,
#     n_jobs=-1,
#     verbose=2
# )

# grid_search.fit(X_train, y_train)

# preds_rf = grid_search.predict(X_val)

# auc = roc_auc_score(y_val, preds_rf)
# print(f"Validation AUC: {auc}")
# print("Best parameters found: ", grid_search.best_params_)

In [30]:
X_train = train['review']
y_train = train['sentiment']
X_test = test['review']

sentences = []
for review in (X_train.tolist() + bonus['review'].tolist()):
    sentences.append(review.split())

fasttext = FastText(
    sentences=sentences,
    vector_size=300,
    window=10,
    min_count=5,
    workers=os.cpu_count(),
    sg=1,
    epochs=10,
    seed=42
)
fasttext.save('models/fasttext_v2.model')

X_train = np.array([sentence_vector(text, fasttext) for text in X_train])
X_test = np.array([sentence_vector(text, fasttext) for text in X_test])

model_rf = RandomForestClassifier(
    n_estimators=2000, 
    criterion='entropy',
    n_jobs=-1,
    random_state=42
)

model_rf.fit(X_train, y_train)

preds_rf = model_rf.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    'id': test_ids,
    'sentiment': preds_rf
})
submission.to_csv('submissions/fasttext_rf_submission.csv', index=False)